# Product Confirmation Workflow

This notebook downloads DIST-ALERT products from S3, unzips them, and runs the confirmation workflow.

In [1]:
import pandas as pd
import shutil
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow
from utils import unzip_dist_s1_prod, wrap_run_sequential_confirmation_of_dist_products_workflow
import multiprocessing

/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tmp_dir =  Path('tmp')
unconfirmed_products_dir =  Path('unconfirmed_products')
confirmed_products_dir =  Path('confirmed_products')

tmp_dir.mkdir(exist_ok=True)
unconfirmed_products_dir.mkdir(exist_ok=True)
confirmed_products_dir.mkdir(exist_ok=True)

In [3]:
# Load the test products CSV
csv_path = Path('dist-s1-events_october-17-2025.csv')
df = pd.read_csv(csv_path)
df.head()

,job_name,zip_url,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,n_workers_for_norm_param_estimation,...,model_source,memory_strategy,batch_size_for_norm_param_estimation,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,1008.556,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-07-30,144,2.5,4,best,none,False
1,durkee_fire_2024__11TMK,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,500.253,4.5,11TMK,1,7,4,...,transformer_optimized,high,32,2024-09-09,42,2.5,4,best,none,False
2,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,1468.343,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-08-23,144,2.5,4,best,none,False
3,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,651.496,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-09-11,71,2.5,4,best,none,False
4,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,667.371,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-06-29,42,2.5,4,best,none,False


In [4]:
def download_file(url, destination_path):
    dst_dir = destination_path.parent
    dst_dir.mkdir(exist_ok=True, parents=True)

    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(destination_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)
    
    return destination_path


In [5]:
download_tasks = []
for _, row in df.iterrows():
    url = row['zip_url']
    filename = Path(url).name
    
    # Customize!!!!
    job_name = row['job_name']
    dist_event = job_name.split('__')[0]
    dst_dir = tmp_dir / dist_event
    zip_path = dst_dir / filename

    download_tasks.append((url, zip_path))

In [ ]:
download_file_p = lambda t: download_file(*t)
with ThreadPoolExecutor(max_workers=10) as executor:
    paths = list(tqdm(executor.map(download_file_p, download_tasks[:]), total=len(download_tasks)))

  0%|                                                                    | 0/876 [00:00<?, ?it/s]

# Unzip

In [ ]:
downloaded_zips = list(Path('tmp/').rglob('*.zip'))
len(downloaded_zips)

In [ ]:
num_processes = 8 
print('Total processes: ', multiprocessing.cpu_count())
print(f"Using {num_processes} processes for unzipping.")

with multiprocessing.Pool(processes=num_processes) as pool:
    results = pool.imap(unzip_dist_s1_prod, downloaded_zips[:])
    for _ in tqdm(results, total=len(downloaded_zips), desc="Unzipping Files"):
        pass

In [ ]:
subdirs = list(Path('unconfirmed_products').rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_tiles_unzipped = list(set([subdir.parent.name for subdir in subdirs]))
mgrs_tiles_unzipped[:3]

In [ ]:
subdirs = list(Path('unconfirmed_products').rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_ts_unzipped_dirs = list(set([subdir.parent for subdir in subdirs]))
mgrs_ts_unzipped_dirs[:3]

In [ ]:
# cleanup_temp = True
# if cleanup_temp:
#     shutil.rmtree(tmp_dir)

# Confirmation

In [ ]:
# %%time

# for mgrs_tile_id in tqdm(mgrs_tiles_unzipped[2:]):
#     # Run the confirmation workflow
#     run_sequential_confirmation_of_dist_products_workflow(
#         unconfirmed_products_dir / mgrs_tile_id, 
#         confirmed_products_dir / mgrs_tile_id
#     )

In [ ]:
Path('.') / mgrs_ts_unzipped_dirs[0].relative_to(unconfirmed_products_dir)

In [ ]:
with multiprocessing.Pool(processes=5) as pool:
    results = pool.imap(wrap_run_sequential_confirmation_of_dist_products_workflow, mgrs_ts_unzipped_dirs[:])
    for _ in tqdm(results, total=len(mgrs_ts_unzipped_dirs), desc="Confirming Products"):
        pass